# Matching Recap and Instrumental Variables

**DS-UA 201: Causal Inference — Lab Notes**

---
# Part 1 — Matching Recap

## 1.1 Key concepts

**The identifying assumption.** Regression with controls and matching both rest on *conditional independence*:

$$S \perp U \mid C$$

Once we fix the controls $C$, treatment $S$ is as good as random with respect to everything else that affects $Y$. Within each subgroup defined by $C$, we effectively have a small natural experiment.

**Exact matching.** If $C$ takes a finite number of values $c_1, \dots, c_K$, compare treated and untreated people *within* each subgroup, then average those comparisons using each subgroup's share of the population:

$$\widehat{ATE}_m = \sum_{k=1}^{K} \Big( \hat E[Y \mid S=1, C=c_k] - \hat E[Y \mid S=0, C=c_k] \Big)\, \hat P(C=c_k)$$

The proof is two steps: the law of iterated expectations splits the ATE into subgroup ATEs, and $S \perp U \mid C$ lets us replace each potential-outcome mean with an observed conditional mean.

**Data requirement.** Every subgroup needs at least one treated and one untreated person (in practice, many of each).

**What regression estimates instead.** With a binary $S$, the coefficient on $S$ in a regression with controls is also a weighted average of subgroup effects, but the weights are not population shares:

$$\alpha_1 = \sum_{k=1}^K E[Y(1,U) - Y(0,U) \mid C=c_k]\; W(C=c_k), \qquad W(C=c_k) \propto P(C=c_k)\,\mathrm{Var}(S \mid C=c_k)$$

Relative to its size, a subgroup gets more weight when treatment status varies a lot inside it ("where the action is"). With a binary $S$, $\mathrm{Var}(S\mid C=c_k) = p_k(1-p_k)$, which peaks when half the subgroup is treated. If effects are the same in every subgroup, the weights don't matter. If effects differ, regression gives a distorted picture of the ATE.

**Curse of dimensionality.** With $J$ binary controls there are $2^J$ subgroups; with continuous controls, essentially one person per value. Many cells end up with only treated or only untreated people, and exact matching becomes impossible.

**Inexact matching.** The fallback. *Nearest-neighbour matching* pairs each treated unit with the untreated unit whose controls are closest under some distance $D(\cdot,\cdot)$, then averages the outcome differences:

$$\widehat{ATE}_{nnm} = \frac{1}{t}\sum_{i=1}^{t}\Big(Y_{i_t} - Y_{i_u}\Big)$$

Strictly, this averages over the treated, so it targets the effect on the treated. Bias grows with the distance between matches, so larger samples help both precision *and* identification. Inexact matching tends to be needed in "kitchen-sink" designs with many controls; designs built on a genuine quasi-random process usually need only a few coarse controls, so exact matching works.

## 1.2 Coded example: when matching and regression disagree

Setup: does attending college ($S$) raise income ($Y$)? Family wealth ($C$) is a confounder: rich families send more children to college *and* have higher incomes regardless. We assume $S \perp U \mid C$, which holds by construction in the simulation.

Two features are built in on purpose:

- **Heterogeneous effects:** college adds \$10,000 for rich students and \$30,000 for poor students.
- **Different treatment variance:** half of rich students attend college ($\mathrm{Var} = 0.25$), but only 10% of poor students do ($\mathrm{Var} = 0.09$).

The true ATE is $0.3 \times 10{,}000 + 0.7 \times 30{,}000 = 24{,}000$.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

In [2]:
rng = np.random.default_rng(42)
n = 5_000

rich = rng.binomial(1, 0.3, n)
college = rng.binomial(1, np.where(rich == 1, 0.5, 0.1))

y0 = 40_000 + 25_000 * rich + rng.normal(0, 5_000, n)
y1 = y0 + np.where(rich == 1, 10_000, 30_000)
income = np.where(college == 1, y1, y0)

mdf = pd.DataFrame({"rich": rich, "college": college, "income": income, "y0": y0, "y1": y1})
mdf.head()

,rich,college,income,y0,y1
0,1,0,55583.357994,55583.357994,65583.357994
1,0,0,45652.389214,45652.389214,75652.389214
2,1,1,80913.624304,70913.624304,80913.624304
3,0,0,37628.165928,37628.165928,67628.165928
4,0,0,42606.379652,42606.379652,72606.379652


In [3]:
true_ate = (mdf["y1"] - mdf["y0"]).mean()
naive = mdf.loc[mdf["college"] == 1, "income"].mean() - mdf.loc[mdf["college"] == 0, "income"].mean()

print(f"True ATE:                     {true_ate:>9,.0f}")
print(f"Naive difference in means:    {naive:>9,.0f}")

True ATE:                        24,112
Naive difference in means:       28,993


**Code explained**

1. `true_ate = (mdf["y1"] - mdf["y0"]).mean()` computes each person's individual effect and averages it. Only possible in a simulation.
2. `naive = ...` is the raw comparison: average income of college-goers minus average income of everyone else. `mdf.loc[condition, "income"]` selects the income column for rows meeting the condition.
3. – 4. The `print` lines report both numbers. Inside the f-string, `>9,.0f` means: right-align in 9 characters, use thousands separators, show no decimals.

The naive estimate (roughly \$28,500) overstates the true ATE (roughly \$24,000) because college-goers are disproportionately rich, and the \$25,000 wealth premium gets credited to college.

In [4]:
cell_means = mdf.groupby(["rich", "college"])["income"].mean().unstack()
cell_effect = cell_means[1] - cell_means[0]
cell_share = mdf["rich"].value_counts(normalize=True).sort_index()

ate_matching = (cell_effect * cell_share).sum()

print(pd.DataFrame({"effect within subgroup": cell_effect, "P(C = c)": cell_share}).round(3))
print(f"\nExact matching ATE:  {ate_matching:,.0f}")

      effect within subgroup  P(C = c)
rich                                  
0                  30363.812     0.706
1                   9906.334     0.294

Exact matching ATE:  24,341


In [5]:
ols = smf.ols("income ~ college + rich", data=mdf).fit()

var_s = mdf.groupby("rich")["college"].var(ddof=0)
reg_weight = cell_share * var_s / (cell_share * var_s).sum()

print(pd.DataFrame({"P(C = c)": cell_share, "Var(S | C = c)": var_s, "regression weight": reg_weight}).round(3))
print(f"\nRegression coefficient on college:            {ols.params['college']:,.0f}")
print(f"Variance-weighted average of subgroup effects: {(cell_effect * reg_weight).sum():,.0f}")

      P(C = c)  Var(S | C = c)  regression weight
rich                                             
0        0.706           0.088              0.457
1        0.294           0.249              0.543

Regression coefficient on college:            19,265
Variance-weighted average of subgroup effects: 19,265


**Code explained**

1. `ols = smf.ols("income ~ college + rich", data=mdf).fit()` runs a regression of income on college while controlling for wealth. `.fit()` estimates it and stores the results in `ols`.
2. `var_s = mdf.groupby("rich")["college"].var(ddof=0)` computes $\mathrm{Var}(S\mid C=c)$ within each wealth group. `ddof=0` uses the population variance formula $p(1-p)$, the version that appears in the weight formula.
3. `reg_weight = cell_share * var_s / (cell_share * var_s).sum()` builds the regression weights: each group's share times its treatment variance, rescaled so the weights sum to 1.
4. The first `print` shows how the weights compare with the population shares. Rich people are 30% of the sample but get about 54% of the regression weight, because their treatment status varies more.
5. The second `print` reports the regression coefficient, about \$19,000, well below the true ATE.
6. The third `print` rebuilds that coefficient by hand as $\sum_k \hat\tau_k \hat W_k$. It matches the regression exactly, confirming the weighting formula.

**Output** Same assumption ($S\perp U\mid C$), same data, two different numbers. Matching weights subgroups by population share and recovers the ATE. Regression over-weights the rich subgroup, which has more treatment variance and the smaller effect. Had the effect been \$30,000 for everyone, both methods would agree.

---
# Part 2 — Instrumental Variables

## 2.1 The problem IV solves

We want the effect of $S$ on $Y$, but other determinants of $Y$ (collected in $U$) are correlated with $S$:

$$S \not\perp U$$

Matching and regression with controls only fix this if we can observe controls $C$ that make $S \perp U \mid C$ true. Often we can't: the confounders are unobserved (ability, motivation, how much free time a student has).

## 2.2 The idea

An **instrument** $Z$ is a third variable that

- affects $S$,
- is as good as random (independent of $U$), and
- affects $Y$ **only through** its effect on $S$.

```
            U
          ↙   ↘
   Z ──→ S ──→ Y          (no arrow Z → Y, no link Z — U)
```

Some of the variation in $S$ comes from $Z$. Since $Z$ is independent of $U$, that slice of variation in $S$ is independent of $U$ too. IV estimates the effect of $S$ on $Y$ using **only that slice**.

## 2.3 Running example: tutoring vouchers and exam results

- $Y = 1$ if the student passes the final exam.
- $S = 1$ if the student attends tutoring.
- $Z = 1$ if the student is randomly sent a voucher for free tutoring, $Z = 0$ if not.

Tutoring is not randomly assigned: motivated students are more likely to sign up *and* more likely to pass anyway, so comparing tutored with untutored students mixes the effect of tutoring with the effect of being motivated. But the vouchers are handed out by lottery, so getting one shifts the chance of attending tutoring without being related to the student.

## 2.4 Notation

We keep the all-causes model for the outcome and add one for the treatment:

$$Y(S, U) \qquad \text{and} \qquad S(Z, V)$$

- $U$: other causes of the outcome.
- $V$: other causes of treatment (things that drive tutoring attendance besides the voucher, like motivation or a busy schedule).
- $S(Z=1,V)$ and $S(Z=0,V)$: potential treatments. They say whether a given student *would* attend tutoring with a voucher and without one. Only one is ever observed.

Note that writing $Y(S,U)$ with no $Z$ inside already encodes the **exclusion restriction**: the voucher affects exam results only through tutoring attendance.

## 2.5 The three LATE assumptions

| Assumption | Formal statement | Meaning in the tutoring example | Testable? |
|---|---|---|---|
| **Exogeneity** | $U, V \perp Z$ | Who gets a voucher is unrelated to students' characteristics or anything else affecting exam results. | Partly: balance on observables. |
| **Relevance** | $S(1,V) \neq S(0,V)$ for some people | The voucher changes at least some students' decision to attend. | Yes: first-stage regression. |
| **Monotonicity** | $S(1,V) \geq S(0,V)$ for everyone | No student who would attend without a voucher is put off by getting one. | Only implications of it. |

Notes on each:

- **Exogeneity** is why the instrument is useful at all. A consequence: the *direct* effect of $Z$ on $Y$ is itself a valid natural-experiment estimate (the "reduced form"). We just don't care about vouchers per se; we care about tutoring. The same logic means random assignment in an experiment with imperfect compliance is an instrument for actual take-up: instead of settling for the intent-to-treat effect, IV recovers the effect on those who complied. It also works for feasible nudges, e.g. randomly offered scholarships as an instrument for college attendance.
- **Relevance** is usually the easy one; if the voucher changes nobody's behaviour, there is no variation to use.
- **Monotonicity** is the subtle one. Suppose the voucher letter makes some students feel singled out as "struggling", so they skip tutoring they would otherwise have attended. On average the voucher still raises attendance, so the data looks fine, but the instrument pushes different people in different directions. Monotonicity fails and the estimate loses its LATE interpretation (Section 2.12 shows how).

## 2.6 Compliance types

Each person's pair of potential treatments puts them into one of four groups:

| Type | $S(Z=0)$ | $S(Z=1)$ | Tutoring example |
|---|---|---|---|
| **Always-taker** | 1 | 1 | Highly motivated students: attend even if they have to pay |
| **Never-taker** | 0 | 0 | Students working full-time: never attend, even for free |
| **Complier** | 0 | 1 | Students who attend only if it's free |
| **Defier** | 1 | 0 | Ruled out by monotonicity |

In this language, relevance says $P(\text{Complier}) > 0$ and monotonicity says $P(\text{Defier}) = 0$.

The target parameter is the **Local Average Treatment Effect**:

$$LATE = E[\,Y(S=1,U) - Y(S=0,U) \mid \text{Complier}\,]$$

IV teaches us nothing about always-takers or never-takers: the instrument never changes their treatment, so it reveals nothing about their counterfactuals. Whether the LATE is policy-relevant depends on whether we care about the compliers. The compliers' response is exactly what a school deciding whether to make tutoring free needs to know; it says little about students who would never show up.

## 2.7 Identification: why the Wald estimator works

### Step 1: constant effects

Take $Y = \alpha_0 + \alpha_1 S + U$ with $E[U]=0$. We cannot assume $\mathrm{Cov}(S,U)=0$, which is the whole problem. Exogeneity does give $\mathrm{Cov}(Z,U)=0$. Take the covariance of both sides with $Z$:

$$\mathrm{Cov}(Z, Y) = \alpha_1\,\mathrm{Cov}(Z, S) + \underbrace{\mathrm{Cov}(Z, U)}_{=\,0}$$

If relevance holds, $\mathrm{Cov}(Z,S) \neq 0$ and we can divide:

$$\boxed{\alpha_1^{IV} = \frac{\mathrm{Cov}(Z,Y)}{\mathrm{Cov}(Z,S)}}$$

This is the **Wald estimator**. Compare OLS, $\mathrm{Cov}(S,Y)/\mathrm{Var}(S)$: IV swaps $S$ for $Z$ in the role of "the thing we correlate with", because $Z$ is clean and $S$ is not.

### Step 2: binary $Z$ turns covariances into differences in means

Dividing top and bottom by $\mathrm{Var}(Z)$, and using the fact that for binary $Z$, $\mathrm{Cov}(Z,X)/\mathrm{Var}(Z) = E[X\mid Z=1]-E[X\mid Z=0]$:

$$\alpha_1^{IV} = \frac{E[Y\mid Z=1] - E[Y\mid Z=0]}{E[S\mid Z=1] - E[S\mid Z=0]} = \frac{\text{reduced form}}{\text{first stage}}$$

### Step 3: heterogeneous effects give the LATE

Now drop the constant-effect assumption. By exogeneity, students with and without vouchers are comparable, so $E[Y\mid Z=1]-E[Y\mid Z=0]$ is the average change in $Y$ caused by giving everyone a voucher. Who actually changes?

- Always-takers and never-takers: their attendance doesn't change, so their outcome doesn't change. They contribute **zero**.
- Compliers: switched from no tutoring to tutoring. Contribute $+\,LATE_C \cdot P(C)$.
- Defiers: switched from tutoring to no tutoring. Contribute $-\,LATE_D \cdot P(D)$.

The same logic applied to $S$ gives the denominator. So in general:

$$\alpha_1^{IV} = \frac{LATE_C\,P(C) - LATE_D\,P(D)}{P(C) - P(D)}$$

Monotonicity sets $P(D)=0$, relevance guarantees $P(C)>0$, and everything collapses:

$$\alpha_1^{IV} = \frac{LATE_C\,P(C)}{P(C)} = LATE$$

The idea in one sentence: **the reduced form is the effect of the instrument on the outcome, diluted by the fact that only compliers react; dividing by the first stage (the share of compliers) undoes the dilution.**

## 2.8 Simulating a population where we know the truth

One function generates the data. It has two optional switches, `defier_share` and `direct_effect`, which are both zero for now. Section 2.12 flips them to break monotonicity and exclusion on purpose.

Built-in truths:

| Type | Share | Baseline pass rate | Effect of tutoring |
|---|---|---|---|
| Always-taker | 20% | 0.60 | +0.05 |
| Complier | 40% | 0.30 | **+0.10** |
| Never-taker | 40% | 0.15 | −0.05 (tutoring eats into hours they need for work) |

So the **true LATE is 0.10**, while the **true ATE is** $0.2(0.05) + 0.4(0.10) + 0.4(-0.05) = 0.03$. Always-takers have the highest baseline pass rate, which is exactly the confounding that sinks the naive comparison.

In [6]:
BASE_PASS     = {"always": 0.60, "complier": 0.30, "defier": 0.30,  "never": 0.15}
TUTORING_EFFECT = {"always": 0.05, "complier": 0.10, "defier": -0.15, "never": -0.05}
PRACTICE_RATE  = {"always": 4.0,  "complier": 2.0,  "defier": 2.0,   "never": 0.5}


def make_population(n, defier_share=0.0, direct_effect=0.0, seed=201):
    rng = np.random.default_rng(seed)
    shares = {"always": 0.2, "complier": 0.4 - defier_share, "defier": defier_share, "never": 0.4}

    df = pd.DataFrame({"type": rng.choice(list(shares), size=n, p=list(shares.values()))})
    df["z"] = rng.binomial(1, 0.5, n)
    df["s"] = np.select(
        [df["type"] == "always", df["type"] == "never", df["type"] == "complier"],
        [1, 0, df["z"]],
        default=1 - df["z"],
    )

    df["practice_tests"] = rng.poisson(df["type"].map(PRACTICE_RATE))
    df["age"] = rng.normal(32, 8, n).round()

    u = rng.uniform(size=n)
    base = df["type"].map(BASE_PASS) + direct_effect * df["z"]
    df["y0"] = (u < base).astype(int)
    df["y1"] = (u < base + df["type"].map(TUTORING_EFFECT)).astype(int)
    df["y"] = np.where(df["s"] == 1, df["y1"], df["y0"])
    return df


df = make_population(20_000)
df.head()

,type,z,s,practice_tests,age,y0,y1,y
0,never,0,0,1,34.0,0,0,0
1,complier,0,0,3,31.0,0,0,0
2,complier,1,1,3,30.0,0,1,1
3,complier,0,0,2,32.0,0,0,0
4,never,0,0,0,52.0,0,0,0


In [7]:
df["effect"] = df["y1"] - df["y0"]

true_ate = df["effect"].mean()
true_late = df.loc[df["type"] == "complier", "effect"].mean()
naive = df.loc[df["s"] == 1, "y"].mean() - df.loc[df["s"] == 0, "y"].mean()

print(df.groupby("type")["effect"].mean().round(3))
print(f"\nTrue ATE:                   {true_ate:.3f}")
print(f"True LATE (compliers):      {true_late:.3f}")
print(f"Naive tutored vs untutored: {naive:.3f}")

type
always      0.049
complier    0.094
never      -0.050
Name: effect, dtype: float64

True ATE:                   0.028
True LATE (compliers):      0.094
Naive tutored vs untutored: 0.323


## 2.9 Checking the assumptions in the data

In real data we can't see `type`, `y0` or `y1`, so from here on we use only `z`, `s`, `y`, `practice_tests` and `age`, exactly what a researcher would have.

### Exogeneity: balance test

If vouchers are handed out at random, students with and without one should look alike on everything determined *before* assignment. A significant difference would be a red flag. Balance only tests observables, though; it can never rule out imbalance in unobservables.

In [8]:
for var in ["practice_tests", "age"]:
    offered = df.loc[df["z"] == 1, var]
    not_offered = df.loc[df["z"] == 0, var]
    t_stat, p_value = stats.ttest_ind(offered, not_offered, equal_var=False)
    print(f"{var:<15} voucher={offered.mean():6.2f}  none={not_offered.mean():6.2f}  "
          f"diff={offered.mean() - not_offered.mean():+.3f}  p={p_value:.3f}")

practice_tests  voucher=  1.82  none=  1.83  diff=-0.005  p=0.858
age             voucher= 32.02  none= 32.09  diff=-0.067  p=0.554


**Code explained**

1. `for var in ["practice_tests", "age"]:` loops over the two pre-assignment characteristics.
2. `offered = df.loc[df["z"] == 1, var]` pulls that variable for students who got a voucher.
3. `not_offered = df.loc[df["z"] == 0, var]` does the same for students who didn't.
4. `t_stat, p_value = stats.ttest_ind(offered, not_offered, equal_var=False)` runs Welch's two-sample t-test, which does not assume equal variances in the two groups. It returns the t-statistic and the p-value for "the means are equal".
5. – 6. `print(...)` reports both group means, their difference, and the p-value. `:<15` left-aligns the variable name in a 15-character column, and `:+.3f` always shows the sign.

Differences should be tiny with large p-values. Practice tests are strongly related to tutoring attendance, but they are balanced *across voucher groups*, which is what matters.


### Relevance: the first stage

Regress the treatment on the instrument. The coefficient on $Z$ is $E[S\mid Z=1]-E[S\mid Z=0]$, which by Step 3 above is the complier share. A common rule of thumb is a first-stage **F-statistic above 10**. Below that the instrument is *weak*: the denominator of the Wald estimator is close to zero, so the estimate becomes noisy and badly behaved.

In [9]:
first_stage_reg = smf.ols("s ~ z", data=df).fit()

print(df.groupby("z")["s"].mean().rename("P(tutoring)").round(3))
print(f"\nFirst-stage coefficient on z: {first_stage_reg.params['z']:.3f}")
print(f"First-stage F-statistic:      {first_stage_reg.fvalue:,.0f}")

z
0    0.202
1    0.603
Name: P(tutoring), dtype: float64

First-stage coefficient on z: 0.401
First-stage F-statistic:      4,001


**Code explained**

1. `first_stage_reg = smf.ols("s ~ z", data=df).fit()` regresses tutoring attendance on the voucher. With a binary outcome this is a linear probability model; the slope is a difference in attendance rates.
2. `print(df.groupby("z")["s"].mean()...)` shows the attendance rate with and without a voucher. `.rename("P(tutoring)")` just labels the output.
3. `first_stage_reg.params["z"]` pulls the slope: about 0.40, i.e. the voucher raises attendance by 40 percentage points. That is the complier share.
4. `first_stage_reg.fvalue` is the regression F-statistic. With a single regressor it equals the squared t-statistic. Here it is in the thousands, nowhere near weak.

### Monotonicity: a testable implication

We can never observe both $S(1,V)$ and $S(0,V)$ for one person, so monotonicity can't be tested directly. But it has an implication: if *nobody* is pushed the "wrong way", the first stage must be **non-negative in every subgroup** we can form:

$$E[S \mid Z=1, \text{group}] \;\geq\; E[S \mid Z=0, \text{group}] \quad \text{for every group}$$

A negative first stage in some subgroup would be clear evidence of defiers. The converse doesn't hold: positive first stages everywhere are reassuring, not proof, because defiers could be outnumbered by compliers inside every subgroup.

In [ ]:
df["practice_group"] = pd.cut(df["practice_tests"], bins=[-1, 0, 2, 4, np.inf], labels=["0", "1-2", "3-4", "5+"])

fs_by_group = df.groupby(["practice_group", "z"], observed=True)["s"].mean().unstack()
fs_by_group.columns = ["P(tutoring | no voucher)", "P(tutoring | voucher)"]
fs_by_group["first stage"] = fs_by_group["P(tutoring | voucher)"] - fs_by_group["P(tutoring | no voucher)"]
fs_by_group.round(3)

,P(tutoring | no voucher),P(tutoring | voucher),first stage
practice_group,,,
0,0.010,0.197,0.187
1-2,0.102,0.623,0.521
3-4,0.409,0.970,0.561
5+,0.794,1.000,0.206


**Code explained**

1. `df["practice_group"] = pd.cut(...)` bins the number of practice tests into four groups. `bins=[-1, 0, 2, 4, np.inf]` creates the intervals (−1,0], (0,2], (2,4], (4,∞), i.e. 0, 1–2, 3–4 and 5+ practice tests; `labels` names them.
2. `fs_by_group = df.groupby(["practice_group", "z"], observed=True)["s"].mean().unstack()` computes the attendance rate for each practice-test group × voucher status, then pivots voucher status into columns. `observed=True` tells pandas to show only combinations that actually occur (needed for categorical columns made by `pd.cut`).
3. `fs_by_group.columns = [...]` renames the columns from `0`/`1` to readable labels.
4. `fs_by_group["first stage"] = ...` takes the difference within each group.
5. `fs_by_group.round(3)` displays the table.

The first stage differs across groups: it is small for students with no practice tests (mostly never-takers, who skip tutoring either way) and for those with 5+ (mostly always-takers, who attend either way), and largest in the middle, where compliers are concentrated. But it is positive in every group, as monotonicity predicts.

## 2.10 Estimating the LATE three ways

All three are algebraically the same number when $Z$ and $S$ are binary and there are no controls. Each one illuminates something different.

### (a) Wald estimator from four means

In [10]:
y_voucher, y_none = df.loc[df["z"] == 1, "y"].mean(), df.loc[df["z"] == 0, "y"].mean()
s_voucher, s_none = df.loc[df["z"] == 1, "s"].mean(), df.loc[df["z"] == 0, "s"].mean()

reduced_form = y_voucher - y_none
first_stage = s_voucher - s_none
wald = reduced_form / first_stage

print(f"Reduced form (effect of voucher on passing):   {reduced_form:.4f}")
print(f"First stage  (effect of voucher on tutoring):  {first_stage:.4f}")
print(f"Wald estimate = reduced form / first stage:    {wald:.4f}")
print(f"True LATE:                                     {true_late:.4f}")

Reduced form (effect of voucher on passing):   0.0418
First stage  (effect of voucher on tutoring):  0.4007
Wald estimate = reduced form / first stage:    0.1044
True LATE:                                     0.0942


**Code explained**

1. `y_voucher, y_none = ...` computes the pass rate with and without a voucher. Python's tuple assignment fills two variables on one line.
2. `s_voucher, s_none = ...` computes the tutoring attendance rate with and without a voucher.
3. `reduced_form = y_voucher - y_none` is the causal effect of the *voucher* on passing. Valid on its own because vouchers are randomly assigned, but it is diluted: 60% of students don't change their behaviour because of the voucher.
4. `first_stage = s_voucher - s_none` is the share of compliers, about 0.40.
5. `wald = reduced_form / first_stage` undoes the dilution: reduced form ≈ 0.04, divided by 0.40, ≈ 0.10.
6. – 9. Print each piece and the true LATE for comparison. The estimate lands within sampling error of 0.10. It does **not** recover the ATE (0.03), and it isn't supposed to.

### (b) Wald estimator as a ratio of covariances

In [11]:
wald_cov = np.cov(df["z"], df["y"])[0, 1] / np.cov(df["z"], df["s"])[0, 1]
print(f"Cov(Z, Y) / Cov(Z, S): {wald_cov:.4f}")

Cov(Z, Y) / Cov(Z, S): 0.1044


**Code explained**

1. `np.cov(df["z"], df["y"])` returns the 2×2 covariance matrix of $Z$ and $Y$; `[0, 1]` picks the off-diagonal entry $\mathrm{Cov}(Z,Y)$. The same is done for $\mathrm{Cov}(Z,S)$, and the two are divided. This is the formula from Step 1 of the identification proof.
2. `print(...)` shows it is identical to the means-based version, as Step 2 promised. The covariance form is the one that generalises to non-binary instruments.

### (c) Two-stage least squares (2SLS)

The name describes the procedure:

1. **First stage:** regress $S$ on $Z$ and keep the fitted values $\hat S$. These are the part of $S$ explained by the instrument, and since $Z \perp U$, $\hat S$ is "clean".
2. **Second stage:** regress $Y$ on $\hat S$.

In [12]:
df["s_hat"] = first_stage_reg.fittedvalues
second_stage = smf.ols("y ~ s_hat", data=df).fit()

print(f"Manual 2SLS coefficient: {second_stage.params['s_hat']:.4f}")
print(f"Manual 2SLS std. error:  {second_stage.bse['s_hat']:.4f}   <- not valid, see below")

Manual 2SLS coefficient: 0.1044
Manual 2SLS std. error:  0.0166   <- not valid, see below


**Code explained**

1. `df["s_hat"] = first_stage_reg.fittedvalues` stores the predicted probability of attending tutoring from the first stage. With a binary $Z$, $\hat S$ takes only two values: the attendance rate with and without a voucher.
2. `second_stage = smf.ols("y ~ s_hat", data=df).fit()` regresses passing on predicted attendance.
3. The coefficient matches the Wald estimate exactly.
4. The standard error is **wrong**. The second-stage regression treats $\hat S$ as if it were data, ignoring that it was itself estimated, and it computes residuals with $\hat S$ instead of the actual $S$. Use the manual version to understand 2SLS, never to report uncertainty.

### Doing it with a library: `linearmodels`

`linearmodels` runs both stages internally and computes correct standard errors. It isn't pre-installed in Colab, so install it once:

In [13]:
%pip install -q linearmodels

Note: you may need to restart the kernel to use updated packages.


In [14]:
from linearmodels.iv import IV2SLS

iv = IV2SLS.from_formula("y ~ 1 + [s ~ z]", data=df).fit(cov_type="robust")
print(iv.summary.tables[1])

ci_low, ci_high = iv.conf_int().loc["s"]
print(f"\nLATE estimate {iv.params['s']:.4f}, 95% CI [{ci_low:.4f}, {ci_high:.4f}]; true LATE {true_late:.4f}")

                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      0.2877     0.0072     40.231     0.0000      0.2737      0.3018
s              0.1044     0.0161     6.4922     0.0000      0.0728      0.1359

LATE estimate 0.1044, 95% CI [0.0728, 0.1359]; true LATE 0.0942


**Code explained**

1. `from linearmodels.iv import IV2SLS` imports the two-stage least squares estimator.
2. `IV2SLS.from_formula("y ~ 1 + [s ~ z]", data=df)` sets up the model. `y ~ 1 + ...` means the outcome is `y` with an intercept (`1`). The bracket `[s ~ z]` is the IV syntax: `s` is the endogenous regressor, instrumented by `z`. Exogenous controls would go outside the brackets, e.g. `y ~ 1 + age + [s ~ z]`.
3. `.fit(cov_type="robust")` estimates it with heteroskedasticity-robust standard errors. Robust errors are the sensible default here because a binary outcome's variance necessarily differs across people.
4. `print(iv.summary.tables[1])` prints just the parameter table (estimates, standard errors, t-stats, p-values, confidence intervals). `tables[0]` would be the header with fit statistics.
5. `ci_low, ci_high = iv.conf_int().loc["s"]` pulls the 95% confidence interval for the coefficient on `s`.
6. `print(...)` reports the estimate, its interval and the truth. The interval should contain 0.10. Notice it is much wider than the OLS interval would be: IV only uses the 40% of variation in $S$ that comes from $Z$, so it pays a precision price for removing bias.

## 2.11 How many compliers? (estimable from the data)

Compliance types are invisible person by person, but under monotonicity their *shares* are identified:

- Anyone who attends **without** a voucher must be an always-taker: $P(AT) = P(S=1\mid Z=0)$
- Anyone who skips tutoring **despite** a voucher must be a never-taker: $P(NT) = P(S=0\mid Z=1)$
- Everyone else is a complier: $P(C) = 1 - P(AT) - P(NT) = $ first stage

This tells you *whose* effect the LATE describes, which belongs in every write-up of an IV result.

In [15]:
estimated = pd.Series({
    "always": s_none,
    "complier": s_voucher - s_none,
    "never": 1 - s_voucher,
})
actual = df["type"].value_counts(normalize=True)

pd.DataFrame({"estimated share": estimated, "true share": actual}).round(3)

,estimated share,true share
always,0.202,0.201
complier,0.401,0.402
never,0.397,0.397


## 2.12 What breaks when an assumption fails

Using the same generator, with breaking assumptions. A very large sample (1,000,000) keeps sampling noise small (about ±0.005), so a large gap between the estimate and the truth is **bias**.

- **Monotonicity fails:** 5% of the population are defiers (taken out of the compliers). From Step 3, the Wald estimator becomes $\frac{0.35(0.10) - 0.05(-0.15)}{0.35 - 0.05} \approx 0.14$, not 0.10.
- **Exclusion fails:** the voucher email *also* includes a free practice exam that raises pass rates directly by 3 points, regardless of whether the student attends tutoring. The reduced form now includes that direct effect, so the Wald estimator picks up a bias of $0.03 / 0.40 = 0.075$. Dividing by the first stage *amplifies* any direct effect, and the weaker the instrument, the worse the amplification.

In [17]:
def wald_estimate(d):
    rf = d.loc[d["z"] == 1, "y"].mean() - d.loc[d["z"] == 0, "y"].mean()
    fs = d.loc[d["z"] == 1, "s"].mean() - d.loc[d["z"] == 0, "s"].mean()
    return rf / fs


scenarios = {
    "All assumptions hold": {},
    "Monotonicity fails (5% defiers)": {"defier_share": 0.05},
    "Exclusion fails (direct effect 0.03)": {"direct_effect": 0.03},
}

rows = []
for name, settings in scenarios.items():
    d = make_population(1_000_000, **settings)
    complier_effect = (d["y1"] - d["y0"])[d["type"] == "complier"].mean()
    rows.append({"scenario": name, "Wald estimate": wald_estimate(d), "true complier LATE": complier_effect})

pd.DataFrame(rows).set_index("scenario").round(3)

,Wald estimate,true complier LATE
scenario,,
All assumptions hold,0.107,0.101
Monotonicity fails (5% defiers),0.150,0.101
Exclusion fails (direct effect 0.03),0.182,0.101


**Line by line**

1. `def wald_estimate(d):` wraps the Wald calculation in a reusable function, since it now runs three times.
2. `rf = ...` is the reduced form for dataset `d`.
3. `fs = ...` is the first stage for dataset `d`.
4. `return rf / fs` returns their ratio.
5. – 9. `scenarios = {...}` maps a readable name to the settings to pass into `make_population`. The empty `{}` means "use the defaults".
10. `rows = []` starts an empty list to collect one result per scenario.
11. `for name, settings in scenarios.items():` loops over the scenarios; `.items()` yields each name together with its settings.
12. `d = make_population(1_000_000, **settings)` generates the scenario's data. `**settings` unpacks the dictionary into keyword arguments, so `{"defier_share": 0.05}` becomes `defier_share=0.05`.
13. `complier_effect = (d["y1"] - d["y0"])[d["type"] == "complier"].mean()` computes the true LATE among compliers, the target IV is supposed to hit.
14. `rows.append({...})` records the scenario name, the Wald estimate and the truth.
15. `pd.DataFrame(rows).set_index("scenario").round(3)` turns the list into a table indexed by scenario.

With all assumptions intact, the estimate and the truth agree to within sampling noise. Each violation produces a gap close to the formulas above. Nothing in the *observed* data flags the problem: the first stage stays positive and balance still holds (vouchers are still randomly assigned). These failures can only be ruled out by arguing from the institutional context, which is why IV papers spend so much space defending their assumptions.

## 2.13 Summary

| | Matching / regression with controls | Instrumental variables |
|---|---|---|
| **Key assumption** | $S \perp U \mid C$ | $Z \perp U, V$; relevance; monotonicity; exclusion built into $Y(S,U)$ |
| **What you need** | The right controls, observed | A variable that moves $S$ as-if randomly and has no other route to $Y$ |
| **Estimand** | ATE (matching) or a variance-weighted average (regression) | LATE: the effect for compliers only |
| **Main risk** | Unobserved confounders | Invalid instrument (exclusion, defiers); weak first stage |
| **Problem** | Curse of dimensionality with many controls | Wider confidence intervals; narrower population |

The core formula, the one worth memorising:

$$\widehat{LATE} = \frac{\text{effect of } Z \text{ on } Y}{\text{effect of } Z \text{ on } S} = \frac{\text{reduced form}}{\text{first stage}}$$
